In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Main libraries
import shap
import numpy as np
import os

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# We will use Polars for data manipulation
import polars as pl
import polars.selectors as cs

# Casting types from time to time to have a better autocompletion
from typing import cast

from models import train_and_explain, ExperimentResults, Species, ModelType, ALL_SPECIES
from analysis import summarize_performance
from config import Ablation, FEATURES_METADATA

import cmocean

CMAP = cmocean.cm.thermal

# Use caching for various results
if not os.path.exists("./cache"):
    os.makedirs("./cache")

In [ ]:
# Pick grouping strategy: "tree_id" or "plot_id" or None
group_col = "tree_id"
# Pick model type: "gbdt" or "lasso"
model_type: ModelType = "gbdt"
# Ablation: "all", "tree-level-only", "plot-level-only", "no-defoliation", "max-defoliation"
ablation: Ablation = "all"

In [ ]:
# Train and explain models for all species
# Set use_temporal_cv = True, to enable HierarchicalTemporalGroup
all_results: dict[Species, ExperimentResults] = {}

for species in ALL_SPECIES:
    all_results[species] = train_and_explain(
        species,
        model_type=model_type,
        group_by=group_col,
        ablation=ablation,
        use_temporal_cv=True,
    )

In [ ]:
# Summarize performance as a table
summarize_performance(
    all_results, ablation=ablation, model_type=model_type, group_col=group_col
)

In [ ]:
from scipy.stats import lognorm
from sklearn.metrics import r2_score

r2_results = []


def inv_transform(y_quantile, shape, loc, scale):
    return lognorm.ppf(y_quantile, shape, loc, scale) - 1


for species, results in all_results.items():
    shape, loc, scale = results.dist_params

    for fold in range(results.num_folds):
        X_train, y_true_train, y_pred_train = results.get_data(fold, split="train")
        X_test, y_true_test, y_pred_test = results.get_data(fold, split="test")

        # Transform to original units for train
        y_true_orig_train = inv_transform(
            np.clip(y_true_train.to_numpy(), 1e-10, 1 - 1e-10), shape, loc, scale
        )
        y_pred_orig_train = inv_transform(
            np.clip(y_pred_train.to_numpy(), 1e-10, 1 - 1e-10), shape, loc, scale
        )

        # Transform to original units for test
        y_true_orig_test = inv_transform(
            np.clip(y_true_test.to_numpy(), 1e-10, 1 - 1e-10), shape, loc, scale
        )
        y_pred_orig_test = inv_transform(
            np.clip(y_pred_test.to_numpy(), 1e-10, 1 - 1e-10), shape, loc, scale
        )

        r2_results.append(
            {
                "species": species,
                "fold": fold,
                "n_train": len(X_train),
                "n_test": len(X_test),
                "quantile_r2 (train)": r2_score(y_true_train, y_pred_train),
                "quantile_r2 (test)": r2_score(y_true_test, y_pred_test),
                "growth_r2 (train)": r2_score(y_true_orig_train, y_pred_orig_train),
                "growth_r2 (test)": r2_score(y_true_orig_test, y_pred_orig_test),
            }
        )

df_r2 = pl.DataFrame(r2_results)

In [ ]:
summary = df_r2.group_by("species").agg(
    [
        # Quantile R²
        pl.col("quantile_r2 (test)").mean().alias("quantile_r2_mean"),
        pl.col("quantile_r2 (test)").std().alias("quantile_r2_std"),
        (
            pl.col("quantile_r2 (test)").std()
            / np.sqrt(pl.col("quantile_r2 (test)").count())
        ).alias("quantile_r2_se"),
        # Growth R²
        pl.col("growth_r2 (test)").mean().alias("growth_r2_mean"),
        pl.col("growth_r2 (test)").std().alias("growth_r2_std"),
        (
            pl.col("growth_r2 (test)").std()
            / np.sqrt(pl.col("growth_r2 (test)").count())
        ).alias("growth_r2_se"),
    ]
)


print(
    f"{'Species':<10} | {'Quantile R² (reported)':<22} | {'Growth R² (original units)':<25}|"
)
print("-" * 65)
for row in summary.iter_rows():
    species = row[0]
    q_mean, q_se = row[1], row[3]
    g_mean, g_se = row[4], row[6]
    quantile_str = f"{q_mean:.3f} ± {q_se:.3f}"
    growth_str = f"{g_mean:.3f} ± {g_se:.3f}"
    print(f"{species:<10} | {quantile_str:<22} | {growth_str:<25} |")